# VeriQuest AI — BCU AI Hackathon 2026

**Colab version** of the full evidence-grounded MCQ pipeline (mirrors `starter_code/run.py`).

Pipeline: question -> multiple targeted search queries -> DuckDuckGo evidence retrieval
-> deduplication -> deterministic evidence ranking -> option scoring -> RAG prompt
-> local <=8B LLM judge -> confidence-weighted decision -> validated CSV export.

Colab doesn't have Ollama available out of the box, so this notebook uses
**Hugging Face `transformers`** to run a local <=8B instruction model
(`Qwen/Qwen2.5-7B-Instruct`, loaded in 4-bit) on Colab's free T4 GPU instead.
The deterministic scoring logic is identical to `run.py` - only the LLM-calling
function differs.

Model rule: the LLM used to answer questions must be <=8B parameters.

## 1. Install packages

Run this cell in Google Colab (with a GPU runtime: Runtime > Change runtime type > T4 GPU).

In [ ]:
!pip install -q pandas ddgs requests tqdm transformers accelerate bitsandbytes

## 2. Imports and configuration

In [ ]:
import json
import re
import time
import urllib.parse
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
from ddgs import DDGS
from tqdm.auto import tqdm

QUESTIONS_FILE = "questions_100.csv"
OUTPUT_FILE = "TEAMNAME_submission.csv"
LIMIT = 5  # set to None for the full 100-question run

ENABLE_WEB_SEARCH = True
MAX_RESULTS_PER_QUERY = 5
MAX_QUERIES_PER_QUESTION = 6
SEARCH_SLEEP_SECONDS = 0.4
TOP_K_EVIDENCE = 5

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # 7.6B parameters - satisfies the <=8B rule
HIGH_CONFIDENCE_THRESHOLD = 0.55

ALLOWED_ANSWERS = {"A", "B", "C", "D", "E", "Unknown"}
OPTION_LETTERS = ["A", "B", "C", "D", "E"]

AUTHORITATIVE_DOMAINS = {
    "wikipedia.org": 1.0,
    "britannica.com": 0.6,
    "imdb.com": 0.5,
    "biography.com": 0.4,
    "bbc.co.uk": 0.3,
    "bbc.com": 0.3,
}

GENERAL_STOPWORDS = {
    "the", "a", "an", "of", "in", "on", "at", "for", "to", "by", "with", "and",
    "or", "is", "are", "was", "were", "be", "been", "being", "that", "this",
    "these", "those", "it", "its", "as", "from", "which", "who", "whom",
    "what", "when", "where", "why", "how", "do", "does", "did", "has", "have",
    "had", "will", "would", "can", "could", "should", "may", "might", "must",
    "not", "no", "yes", "than", "then", "also", "into", "about", "over",
    "under", "between", "among", "during", "after", "before", "up", "down",
    "out", "off", "again", "further", "each", "other", "some", "such", "only",
    "own", "same", "so", "too", "very", "just", "now", "according",
}

QUESTION_LEADING_STOPWORDS = {
    "what", "who", "when", "where", "which", "why", "how", "is", "are", "was",
    "were", "do", "does", "did", "the", "a", "an", "of", "in", "on", "at",
    "for", "to", "by", "with", "according", "wikipedia", "and", "or",
}

## 3. Upload the question file

Upload `questions_100.csv` to this Colab session.

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

## 4. Load questions

In [ ]:
def load_questions(path: str) -> pd.DataFrame:
    """Load the official question file."""
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"Could not find {path}. Upload the file or update QUESTIONS_FILE.")

    questions = pd.read_csv(file_path)
    required_columns = {"question_no", "question"}
    missing = required_columns - set(questions.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    return questions


def get_options(row: pd.Series) -> Dict[str, str]:
    options = {}
    for letter in OPTION_LETTERS:
        value = row.get(letter, "")
        if pd.notna(value) and str(value).strip():
            options[letter] = str(value).strip()
    return options

## 5. Text utilities

Shared helpers used by query building, evidence ranking, and option scoring.

In [ ]:
def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def tokenize(text: str) -> List[str]:
    words = re.findall(r"[a-zA-Z0-9]+", str(text).lower())
    return [w for w in words if w not in GENERAL_STOPWORDS and len(w) > 1]


def extract_main_entity(question: str) -> str:
    """Heuristically pull the main proper-noun entity out of a question.

    Prefers quoted text (titles, names in quotes), otherwise picks the
    longest run of consecutive capitalised words that aren't question
    stopwords - this captures things like "P. Padmarajan" or
    "United States Marine Forces Special Operations Command".
    """
    text = question.strip().rstrip("?").strip()

    quote_match = re.search(r'["\u201c]([^"\u201d]{2,60})["\u201d]', text)
    if quote_match:
        return quote_match.group(1).strip()

    words = text.split()
    candidates: List[List[str]] = []
    current: List[str] = []
    for word in words:
        bare = re.sub(r"[^\w.'-]", "", word)
        if not bare:
            if current:
                candidates.append(current)
                current = []
            continue
        if bare[0].isupper() and bare.lower() not in QUESTION_LEADING_STOPWORDS:
            current.append(bare)
        else:
            if current:
                candidates.append(current)
                current = []
    if current:
        candidates.append(current)

    if candidates:
        best = max(candidates, key=lambda c: len(" ".join(c)))
        return " ".join(best)
    return ""


def normalize_url(url: str) -> str:
    if not url:
        return ""
    try:
        parsed = urllib.parse.urlsplit(url)
        netloc = parsed.netloc.lower()
        if netloc.startswith("www."):
            netloc = netloc[4:]
        path = parsed.path.rstrip("/")
        return f"{netloc}{path}"
    except Exception:
        return url.lower()


def fingerprint_text(text: str) -> str:
    norm = re.sub(r"[^a-z0-9 ]", "", text.lower())
    norm = re.sub(r"\s+", " ", norm).strip()
    return norm[:160]


def extract_numbers(text: str) -> set:
    """Extract numeric substrings (including decimals) for robust fact matching.

    Wording around numbers varies a lot ("22.8-mile-long" vs "22.8 miles"),
    so matching the digits directly is far more reliable than word-token
    overlap for length/date/year/population-style answers.
    """
    return set(re.findall(r"\d+\.?\d*", text))

## 6. Build search queries

Multiple short, targeted queries instead of one giant query containing the question plus every option (the original starter's weakness).

In [ ]:
def pick_distinctive_keywords(options: Dict[str, str], limit: int = 2) -> List[str]:
    """Find keywords that appear in exactly one option (i.e. distinguish it
    from the others), so we can search for the option's most specific term
    instead of dumping every option into one query."""
    token_counts: Counter = Counter()
    option_tokens: Dict[str, List[str]] = {}
    for letter, text in options.items():
        toks = tokenize(text)
        option_tokens[letter] = toks
        for t in set(toks):
            token_counts[t] += 1

    candidates = []
    for toks in option_tokens.values():
        distinctive = sorted({t for t in toks if token_counts[t] == 1}, key=len, reverse=True)
        if distinctive:
            candidates.append(distinctive[0])

    candidates = sorted(set(candidates), key=len, reverse=True)
    return candidates[:limit]


def build_search_queries(row: pd.Series) -> List[str]:
    """Build 3-6 short, targeted search queries instead of one giant query
    containing the question plus every option."""
    question = normalize_whitespace(str(row.get("question", "")))
    options = get_options(row)
    entity = extract_main_entity(question)

    queries: List[str] = []

    if question:
        queries.append(question.rstrip("?"))

    if entity:
        queries.append(f'"{entity}"')

        question_tokens = tokenize(question)
        entity_tokens = set(tokenize(entity))
        property_tokens = [t for t in question_tokens if t not in entity_tokens]
        if property_tokens:
            queries.append(f"{entity} {' '.join(property_tokens[:4])}")

        queries.append(f"{entity} wikipedia")

    base_for_option_query = entity if entity else question.rstrip("?")
    for term in pick_distinctive_keywords(options, limit=2):
        queries.append(f"{base_for_option_query} {term}")

    queries.append(question)

    deduped: List[str] = []
    seen = set()
    for q in queries:
        q_norm = normalize_whitespace(q)
        key = q_norm.lower()
        if not q_norm or key in seen:
            continue
        seen.add(key)
        deduped.append(q_norm)

    if len(deduped) < 3:
        for text in options.values():
            first_word = text.split()[0] if text.split() else ""
            q = normalize_whitespace(f"{base_for_option_query} {first_word}")
            key = q.lower()
            if q and key not in seen:
                seen.add(key)
                deduped.append(q)
            if len(deduped) >= 3:
                break

    return deduped[:MAX_QUERIES_PER_QUESTION]

## 7. Retrieve evidence from DuckDuckGo

Searches multiple queries per question, deduplicates by URL and by a normalized title+snippet fingerprint, and never crashes the run on a failed search.

In [ ]:
_ddgs_warned = False


def search_duckduckgo(query: str, max_results: int = MAX_RESULTS_PER_QUERY) -> List[Dict[str, str]]:
    if not ENABLE_WEB_SEARCH:
        return []

    results_out = []
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=max_results)
            for item in results:
                results_out.append(
                    {
                        "title": item.get("title", "") or "",
                        "snippet": item.get("body", "") or "",
                        "url": item.get("href", "") or "",
                        "query": query,
                    }
                )
    except Exception as error:
        print(f"[Warning] DuckDuckGo search failed for query '{query}': {error}")

    time.sleep(SEARCH_SLEEP_SECONDS)
    return results_out


def gather_evidence(queries: List[str]) -> List[Dict[str, str]]:
    all_evidence: List[Dict[str, str]] = []
    for query in queries:
        all_evidence.extend(search_duckduckgo(query))
    return all_evidence


def dedupe_evidence(evidence: List[Dict[str, str]]) -> List[Dict[str, str]]:
    seen_urls = set()
    seen_fingerprints = set()
    deduped = []
    for item in evidence:
        url_key = normalize_url(item.get("url", ""))
        fingerprint = fingerprint_text(f"{item.get('title', '')} {item.get('snippet', '')}")
        if url_key and url_key in seen_urls:
            continue
        if fingerprint and fingerprint in seen_fingerprints:
            continue
        if url_key:
            seen_urls.add(url_key)
        if fingerprint:
            seen_fingerprints.add(fingerprint)
        deduped.append(item)
    return deduped

## 8. Rank evidence

Deterministic keyword/phrase/source scoring - no embeddings needed at this scale, and it's fully explainable.

In [ ]:
def score_evidence_item(
    item: Dict[str, str],
    question_tokens: set,
    option_token_sets: Dict[str, set],
    entity_lower: str,
    exact_phrases: List[str],
) -> float:
    title = item.get("title", "") or ""
    snippet = item.get("snippet", "") or ""
    text = f"{title} {snippet}"
    text_lower = text.lower()
    tokens = set(tokenize(text))

    score = 0.0

    if question_tokens:
        overlap = len(tokens & question_tokens) / len(question_tokens)
        score += overlap * 2.0

    best_option_overlap = 0.0
    for opt_tokens in option_token_sets.values():
        if opt_tokens:
            overlap = len(tokens & opt_tokens) / len(opt_tokens)
            best_option_overlap = max(best_option_overlap, overlap)
    score += best_option_overlap * 1.5

    if entity_lower and entity_lower in text_lower:
        score += 1.5

    title_tokens = set(tokenize(title))
    if question_tokens:
        title_overlap = len(title_tokens & question_tokens) / len(question_tokens)
        score += title_overlap * 1.0

    for phrase in exact_phrases:
        phrase = phrase.strip().lower()
        if len(phrase) > 6 and phrase in text_lower:
            score += 1.0
            break

    url = (item.get("url", "") or "").lower()
    for domain, bonus in AUTHORITATIVE_DOMAINS.items():
        if domain in url:
            score += bonus
            break

    return score


def rank_evidence(
    row: pd.Series,
    evidence: List[Dict[str, str]],
    options: Dict[str, str],
    top_k: int = TOP_K_EVIDENCE,
) -> List[Dict[str, str]]:
    question = str(row.get("question", ""))
    question_tokens = set(tokenize(question))
    option_token_sets = {letter: set(tokenize(text)) for letter, text in options.items()}
    entity = extract_main_entity(question)
    entity_lower = entity.lower()
    exact_phrases = list(options.values()) + ([entity] if entity else [])

    scored = []
    for item in evidence:
        relevance = score_evidence_item(item, question_tokens, option_token_sets, entity_lower, exact_phrases)
        item_copy = dict(item)
        item_copy["relevance_score"] = round(relevance, 4)
        scored.append(item_copy)

    scored.sort(key=lambda x: x["relevance_score"], reverse=True)
    return scored[:top_k]

## 9. Score options A-E and compute confidence

Each option gets a deterministic support score from the ranked evidence, independent of the LLM. Confidence combines the top score, the gap to the runner-up, evidence quantity, and exact-match flags - it decides whether to trust this deterministic scorer or the LLM judge.

In [ ]:
def score_options(
    options: Dict[str, str], ranked_evidence: List[Dict[str, str]], question: str = ""
) -> Tuple[Dict[str, float], Dict[str, bool]]:
    """Score how strongly the ranked evidence supports each option.

    Returns (option_scores, exact_match_flags), e.g.
    {"A": 0.32, "B": 0.81, "C": 0.14, "D": 0.09, "E": 0.18}.
    """
    raw_scores = {letter: 0.0 for letter in options}
    exact_match = {letter: False for letter in options}

    # Only numeric tokens shared with the question (e.g. a route number
    # repeated in every search result) are excluded from option overlap.
    # Shared CONTENT WORDS are kept - they're often a legitimate hint baked
    # into the question itself, not noise.
    question_tokens = set(tokenize(question))
    shared_numeric_noise = {t for t in question_tokens if t.isdigit()}
    option_token_sets = {
        letter: set(tokenize(text)) - shared_numeric_noise for letter, text in options.items()
    }
    option_numbers = {letter: extract_numbers(text) for letter, text in options.items()}

    token_counts: Counter = Counter()
    for toks in option_token_sets.values():
        for t in toks:
            token_counts[t] += 1
    distinctive_sets = {
        letter: {t for t in toks if token_counts[t] == 1} for letter, toks in option_token_sets.items()
    }

    for rank, item in enumerate(ranked_evidence):
        weight = 1.0 / (rank + 1)
        title = item.get("title", "") or ""
        snippet = item.get("snippet", "") or ""
        raw_text = f"{title} {snippet}"
        text_lower = raw_text.lower()
        ev_tokens = set(tokenize(raw_text)) - shared_numeric_noise
        ev_numbers = extract_numbers(raw_text)
        title_lower = title.lower()

        for letter, opt_tokens in option_token_sets.items():
            if not opt_tokens:
                continue

            overlap = len(ev_tokens & opt_tokens) / len(opt_tokens)
            raw_scores[letter] += overlap * weight * 1.0

            # Lower weight than the base overlap fraction on purpose: a
            # "distinctive" single-word match can be a false signal when it
            # shows up in unrelated evidence text. The overlap fraction
            # above is the more reliable primary signal.
            dist_overlap = len(ev_tokens & distinctive_sets[letter])
            raw_scores[letter] += dist_overlap * weight * 0.3

            option_text_lower = options[letter].lower()
            phrase_chunk = option_text_lower[:60]
            if len(phrase_chunk) > 8 and phrase_chunk in text_lower:
                raw_scores[letter] += 2.0 * weight
                exact_match[letter] = True

            if option_text_lower[:30] and option_text_lower[:30] in title_lower:
                raw_scores[letter] += 1.0 * weight

            numbers = option_numbers[letter]
            if numbers and numbers.issubset(ev_numbers):
                raw_scores[letter] += 2.5 * weight
                exact_match[letter] = True

    max_score = max(raw_scores.values()) if raw_scores else 0.0
    if max_score > 0:
        normalized = {letter: round(val / max_score, 4) for letter, val in raw_scores.items()}
    else:
        normalized = {letter: 0.0 for letter in raw_scores}

    return normalized, exact_match


def best_deterministic_answer(option_scores: Dict[str, float]) -> str:
    if not option_scores:
        return "Unknown"
    best_letter = max(option_scores, key=option_scores.get)
    if option_scores[best_letter] <= 0:
        return "Unknown"
    return best_letter


def compute_confidence(
    option_scores: Dict[str, float],
    ranked_evidence: List[Dict[str, str]],
    exact_match_flags: Dict[str, bool],
) -> float:
    """Gap is weighted heavily (0.7) on purpose: a high top score that is
    TIED with another option means the scorer can't actually discriminate
    between them, so a tie gets an explicit penalty."""
    if not option_scores or max(option_scores.values()) <= 0:
        return 0.0

    sorted_scores = sorted(option_scores.values(), reverse=True)
    best = sorted_scores[0]
    second = sorted_scores[1] if len(sorted_scores) > 1 else 0.0
    gap = best - second
    base = 0.3 * best + 0.7 * gap

    tie_count = sum(1 for v in option_scores.values() if abs(v - best) < 1e-6)
    if tie_count > 1:
        base *= 0.3

    evidence_bonus = min(0.15, 0.03 * len(ranked_evidence))
    exact_bonus = 0.15 if any(exact_match_flags.values()) else 0.0

    confidence = max(0.0, min(1.0, base + evidence_bonus + exact_bonus))
    return round(confidence, 4)

## 10. Build the RAG prompt

The LLM acts as an evidence judge, not a memory-based guesser: it sees the question, options, ranked evidence (title/snippet/URL), and the deterministic option scores, then must return exactly one token.

In [ ]:
def build_prompt(
    row: pd.Series,
    options: Dict[str, str],
    ranked_evidence: List[Dict[str, str]],
    option_scores: Dict[str, float],
) -> str:
    lines = [
        "You are an evidence-based multiple-choice exam judge.",
        "Base your answer on the evidence below. If evidence is available, prioritise it",
        "over your own prior knowledge. If no evidence was retrieved, use your best general",
        "knowledge, but answer Unknown if you are not reasonably confident.",
        "",
        f"Question: {row.get('question', '')}",
        "",
        "Options:",
    ]
    for letter in OPTION_LETTERS:
        if letter in options:
            lines.append(f"{letter}. {options[letter]}")

    lines.append("")
    lines.append("Evidence (ranked by relevance):")
    if ranked_evidence:
        for i, item in enumerate(ranked_evidence, start=1):
            lines.append(f"[{i}] Title: {item.get('title', '')}")
            lines.append(f"    Snippet: {item.get('snippet', '')}")
            lines.append(f"    URL: {item.get('url', '')}")
    else:
        lines.append("(No evidence retrieved.)")

    lines.append("")
    lines.append("Deterministic evidence-overlap scores per option (higher = more textual support):")
    for letter in OPTION_LETTERS:
        if letter in option_scores:
            lines.append(f"  {letter}: {option_scores[letter]}")

    lines.append("")
    lines.append("Instructions: choose the option letter most strongly supported by the evidence above.")
    lines.append("Respond with EXACTLY ONE TOKEN: A, B, C, D, E, or Unknown. No explanation, no punctuation.")
    lines.append("Answer:")
    return "\n".join(lines)

## 11. Load the local <=8B model

Ollama isn't available in Colab by default, so this notebook loads a <=8B Hugging Face instruct model directly with `transformers`, 4-bit quantized so it fits comfortably on a free T4 GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
)
print(f"Loaded {MODEL_NAME} (<=8B parameters) in 4-bit.")

In [ ]:
DISABLE_LLM = False


def call_local_llm(prompt: str) -> Optional[str]:
    """Call the local <=8B model. Returns None (never raises) on failure so
    the pipeline can fall back to the deterministic scorer."""
    if DISABLE_LLM:
        return None
    try:
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                inputs,
                max_new_tokens=8,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = output_ids[0][inputs.shape[-1]:]
        return tokenizer.decode(generated, skip_special_tokens=True).strip()
    except Exception as error:
        print(f"[Warning] Local LLM generation failed: {error}")
        return None

## 12. Answer validation

In [ ]:
def clean_answer(answer) -> str:
    """Validate and normalise a raw model/deterministic answer. Never blank."""
    if answer is None:
        return "Unknown"

    text = str(answer).strip()
    if not text:
        return "Unknown"

    if text.strip().lower() == "unknown":
        return "Unknown"

    compact = re.sub(r"[^A-Za-z]", "", text)
    if len(compact) == 1 and compact.upper() in OPTION_LETTERS:
        return compact.upper()

    match = re.search(r"answer[^A-Za-z]{0,10}\b([A-E])\b", text, re.IGNORECASE)
    if match:
        return match.group(1).upper()

    if re.search(r"unknown", text, re.IGNORECASE):
        return "Unknown"

    match = re.search(r"\b([A-E])\b", text)
    if match:
        return match.group(1)

    return "Unknown"


def decide_final_answer(deterministic_answer: str, llm_answer: str, confidence: float) -> Tuple[str, str]:
    """Use confidence to decide whether to trust the deterministic evidence
    scorer or the LLM judge."""
    if confidence >= HIGH_CONFIDENCE_THRESHOLD and deterministic_answer != "Unknown":
        return deterministic_answer, "deterministic-high-confidence"
    if llm_answer and llm_answer != "Unknown":
        return llm_answer, "llm"
    if deterministic_answer != "Unknown":
        return deterministic_answer, "deterministic-fallback"
    return "Unknown", "unknown"

## 13. Run the full pipeline

`LIMIT` is set above for quick testing - set `LIMIT = None` for the full 100-question run.

In [ ]:
def answer_question(row: pd.Series) -> Dict:
    options = get_options(row)
    queries = build_search_queries(row)

    raw_evidence = gather_evidence(queries)
    deduped_evidence = dedupe_evidence(raw_evidence)
    ranked = rank_evidence(row, deduped_evidence, options)

    option_scores, exact_flags = score_options(options, ranked, str(row.get("question", "")))
    deterministic_answer = best_deterministic_answer(option_scores)
    confidence = compute_confidence(option_scores, ranked, exact_flags)

    prompt = build_prompt(row, options, ranked, option_scores)
    llm_raw = call_local_llm(prompt)
    llm_answer = clean_answer(llm_raw) if llm_raw is not None else "Unknown"

    final_answer, decision_reason = decide_final_answer(deterministic_answer, llm_answer, confidence)
    final_answer = clean_answer(final_answer)

    return {
        "question_no": row.get("question_no"),
        "question": row.get("question"),
        "answer": final_answer,
        "confidence": confidence,
        "llm_answer": llm_answer,
        "deterministic_answer": deterministic_answer,
        "decision_reason": decision_reason,
        "search_queries": queries,
        "option_scores": option_scores,
        "ranked_evidence": ranked,
    }


questions = load_questions(QUESTIONS_FILE)
if LIMIT is not None:
    questions = questions.head(LIMIT)

print(f"Answering {len(questions)} question(s) using model='{MODEL_NAME}'...")

results = []
for _, row in tqdm(questions.iterrows(), total=len(questions), desc="Answering questions"):
    results.append(answer_question(row))

predictions = pd.DataFrame([{"question_no": r["question_no"], "answer": r["answer"]} for r in results])
predictions.head(10)

## 14. Validate and export the submission

Ensures the columns are exactly `question_no,answer`, every answer is in `{A,B,C,D,E,Unknown}`, and warns if the row count isn't 100 (e.g. because `LIMIT` was used).

In [ ]:
def validate_submission(preds: pd.DataFrame, limit: Optional[int]) -> None:
    problems = []
    for _, p in preds.iterrows():
        answer = p["answer"]
        if answer is None or str(answer).strip() == "":
            problems.append(f"Blank answer for question_no={p['question_no']}")
        elif answer not in ALLOWED_ANSWERS:
            problems.append(f"Invalid answer '{answer}' for question_no={p['question_no']}")

    if problems:
        raise ValueError("Submission validation failed:\n" + "\n".join(problems))

    if limit is None and len(preds) != 100:
        print(f"[Warning] Expected 100 rows but produced {len(preds)}. Check questions_100.csv.")
    elif limit is not None:
        print(f"[Info] Produced {len(preds)} row(s) because LIMIT={limit} was used. Set LIMIT=None for the full submission.")

    print("Submission validation passed: columns are 'question_no,answer' and all answers are valid.")


validate_submission(predictions, LIMIT)
predictions.to_csv(OUTPUT_FILE, index=False)
print(f"Saved submission file to: {OUTPUT_FILE}")

## 15. Download the submission file

In [ ]:
files.download(OUTPUT_FILE)

## Notes

- **Model rule compliance:** `Qwen/Qwen2.5-7B-Instruct` has 7.6B parameters, satisfying the <=8B limit. This is the only model used to generate answers.
- **Why this differs from `run.py`:** the canonical, fully-tested implementation is `starter_code/run.py`, which uses Ollama + `llama3.1:8b` and is what generated the official `APEXMIND_submission.csv`. This notebook ports the identical deterministic logic (search query building, evidence ranking, option scoring, confidence) for participants who prefer Colab over a local Ollama setup, swapping only the LLM-calling step to Hugging Face `transformers` since Ollama isn't available in Colab by default.
- **GPU required:** make sure Colab's runtime is set to a GPU (T4 is sufficient for a 4-bit 7B model) before running cell 11.
- Internet-assisted search (DuckDuckGo) is used throughout, consistent with the hackathon rules.